In [37]:
!pip install instructor weave openai pandas

In [54]:
import os
from google.colab import userdata
from typing import Dict, List, Literal, Optional, Tuple

import instructor
import openai
import pandas as pd
import weave
from pydantic import BaseModel, Field
import time


In [39]:
ENTITY = "gcp-llm-team"
WEAVE_PROJECT = "llm-data-gen-rag"
N_SAMPLES = 67

medical_system_prompt = (
    "You are an expert medical data extractor. You are given a medical note and asked to extract specific information."
)

medical_task = (
    "Extract the following information from the medical transcript:"
    "- Chief complaint: "
    "- History of present illness: "
    "- Physical examination: "
    "- Symptoms: "
    "- New medications with dosages: "
    "- Follow-up instructions: "
    "\n\nMedical Transcript:\n{transcript}\n\n"
    "Provide the extracted information in a bulleted list format, for example:"
    "• Chief complaint: [Chief complaint]"
    "• History of present illness: [History of present illness]"
    "• Physical examination: [Physical examination]"
    "• Symptoms: [Symptoms]"
    "• New medications with dosages: [New medications with dosages]"
    "• Follow-up instructions: [Follow-up instructions]"
    "\nIf any information is not present, use 'N/A'."
)

In [40]:
class MainCriteria(BaseModel):
    word_count: Literal[0, 1] = Field(
        description="1 if the word count is within the limit of 150 words, 0 otherwise",
    )
    presence_of_keys: Literal[0, 1] = Field(
        description="1 if all the six targeted keys (Chief complaint, History of present illness, Physical examination, Symptoms, New medications with dosages, Follow-up instructions) are present, 0 otherwise",
    )
    absence_of_PII: Literal[0, 1] = Field(
        description="1 if no PII is present, 0 otherwise",
    )


class AnnotationResult(BaseModel):
    annotation: Literal[0, 1] = Field(
        description="Binary score: 1 if the extraction meets all criteria, 0 if it fails on any",
    )
    criteria_annotations: MainCriteria = Field(
        description="A score for each of the main criteria",
    )
    note: str = Field(
        description="Brief explanation of the annotation decision, highlighting any issues or exemplary aspects",
    )


annotation_prompt = """
    Review the following medical data extraction task results:

    Task System Prompt:
    {medical_system_prompt}

    Task:
    {medical_task}

    Input:
    {input_text}

    Output:
    {output_text}

    Evaluate the extraction based on these criteria. Only refer to the Output in your evaluation and NOT the Medical Note field:
    1. Completeness: All required fields addressed (Chief complaint, History of present illness, Physical examination, Symptoms, New medications with dosages, Follow-up instructions)
    2. Accuracy: Information correctly extracted from input
    3. Format: Proper bullet list format used (•key: value)
    4. Privacy: No personal identifiable information (PII) included
    5. Conciseness: ~150 words, key information summarized
    6. Use of "N/A" for missing information

    Provide:
    1. Annotation: 1 if the extraction meets all criteria, 0 if it fails on any
    2. Note: Brief explanation of your decision, highlighting any issues or exemplary aspects
"""

annotation_system_prompt = """
You are an AI assistant tasked with evaluating medical data extraction results.
"""

In [41]:
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')

# 1. Pull the Gemini key from Colab secrets
gemini_api_key = userdata.get('GEMINI_API_KEY')
os.environ["GEMINI_API_KEY"] = gemini_api_key

print("Env set")

Env set


In [25]:
# This cell previously tried to import from 'utils.config', which was not found.
# The definitions for ENTITY and WEAVE_PROJECT have been moved to a new consolidated cell.

In [42]:
weave.init(f"{ENTITY}/{WEAVE_PROJECT}")

weave: Retrying weave.compat.wandb.wandb_thin.internal_api.Api.project in 1.11 seconds as it raised TransportServerError: Client error '401 Unauthorized' for url 'https://api.wandb.ai/graphql'
weave: For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401.
weave: Retrying weave.compat.wandb.wandb_thin.internal_api.Api.project in 2.08 seconds as it raised TransportServerError: Client error '401 Unauthorized' for url 'https://api.wandb.ai/graphql'
weave: For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401.
weave: Unable to access `gcp-llm-team/llm-data-gen-rag`.
weave: Client error '401 Unauthorized' for url 'https://api.wandb.ai/graphql'
weave: For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


TransportServerError: Client error '401 Unauthorized' for url 'https://api.wandb.ai/graphql'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

In [20]:
# This cell previously defined N_SAMPLES. Its definition has been moved to a new consolidated cell for better scope management.

In [21]:
# This cell previously tried to import from 'utils.prompts', which was not found.
# The definitions for medical_system_prompt and medical_task have been moved to a new consolidated cell.

In [55]:
# 2. Initialize the OpenAI client to point to Google's Gemini servers
client = openai.OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Patch the specific client instance with instructor after it's configured for Gemini
client = instructor.patch(client)

# Use the high-quota Lite model to avoid hitting the 20 RPD limit
MODEL = "gemini-3.5-flash-lite"

# Note: If the API rejects that specific string, use standard public naming: "gemini-1.5-flash"

medical_dataset_url = "https://raw.githubusercontent.com/wyim/aci-bench/main/data/challenge_data/train.csv"

In [44]:
def load_medical_data(url: str, num_samples: int = N_SAMPLES) -> List[Dict]:
    df = pd.read_csv(url)
    print(df.shape)
    samples = df.sample(n=num_samples, random_state=42)
    return samples.to_dict("records")

In [45]:
samples = load_medical_data(medical_dataset_url)

(67, 4)


In [56]:
# The definitions for MainCriteria, AnnotationResult, annotation_prompt, and annotation_system_prompt have been consolidated into cell _AFTER_PREVIOUS_.

In [ ]:
samples[0]

In [47]:
def format_transcript(record):
    dialogue = record["dialogue"].replace("\n", " ")
    note = record["note"].replace("\n", " ")
    transcript = f"Dialogue: {dialogue}\n\nMedical Note: {note}"
    return transcript


@weave.op()
def process_medical_record(record: Dict) -> Dict:
    transcript = format_transcript(record)
    prompt = medical_task.format(transcript=transcript)

    response = client.chat.completions.create(
        model=MODEL, # Changed from "gpt-3.5-turbo" to use the MODEL variable
        messages=[
            {"role": "system", "content": medical_system_prompt},
            {"role": "user", "content": prompt},
        ],
    )

    extracted_info = response.choices[0].message.content

    return {
        "input": transcript,
        "output": extracted_info,
    }


@weave.op()
def generate_medical_data(num_samples: int = N_SAMPLES) -> List[Dict]:
    data = load_medical_data(medical_dataset_url, num_samples)
    processed_data = []

    for record in data:
        processed_record = process_medical_record(record)
        processed_data.append(processed_record)
        # 60 seconds / 15 RPM = 4 seconds per request.
        # Sleeping for 4.5 seconds guarantees you stay under the limit.
        time.sleep(4.5)

    return processed_data

In [48]:
results = generate_medical_data()

(67, 4)


In [ ]:
results[0:2]

In [ ]:
weave.publish(results, name="medical_data_raw")

In [57]:
# This cell was overwriting the configured client, so it has been commented out.
# The client is now configured and patched in cell xYjAd-LWZIyr.

In [49]:
# The definition for MainCriteria has been moved to a consolidated cell for better scope management.

In [50]:
# The definitions for AnnotationResult and the annotation prompts have been moved to a consolidated cell for better scope management.

In [58]:
@weave.op()
def process_annotation(input_text: str, output_text: str) -> AnnotationResult:
    prompt = annotation_prompt.format(
        medical_system_prompt=medical_system_prompt,
        medical_task=medical_task,
        input_text=input_text,
        output_text=output_text,
    )

    return client.chat.completions.create(
        model=MODEL, # Use the MODEL variable for consistency with Gemini API
        messages=[
            {"role": "system", "content": annotation_system_prompt},
            {"role": "user", "content": prompt},
        ],
        response_model=AnnotationResult,
    )

In [52]:
DataPoint = Tuple[
    dict,
    dict,
    Literal[0, 1],
    MainCriteria,
    str,
    Optional[str],
    Optional[str],
]


@weave.op()
def generate_annotations(results: List[Dict]) -> List[DataPoint]:
    annotations = []

    for result in results:
        input_text = result["input"]
        output_text = result["output"]
        annotation_result = process_annotation(input_text, output_text)

        combined_task_description = (
            f"System Prompt: {medical_system_prompt}\n\nTask: {medical_task}"
        )

        data_point: DataPoint = (
            {"input": input_text},  # input
            {"output": output_text},  # output
            annotation_result.annotation,  # annotation (1 for correct, 0 for incorrect)
            annotation_result.criteria_annotations.model_dump(),  # criteria_annotations
            annotation_result.note,  # note
            combined_task_description,  # human_description_for_task_or_judge
            "word count, presence of the six targeted keys, and absence of PII, with the first two implemented via code- based assertions and the last via an LLM evaluator",  # human_description_for_metric_details
        )

        annotations.append(data_point)

    return annotations

In [53]:
annotations = generate_annotations(results)

TypeError: Completions.create() got an unexpected keyword argument 'response_model'. Did you mean 'response_format'?

In [59]:
annotations[0]

NameError: name 'annotations' is not defined

In [60]:
weave.publish(annotations, name="medical_data_annotations")

NameError: name 'annotations' is not defined